# Part 3: Steady state analysis

### Part 3.1: Steady state finding

We will start by showing how to find model *steady states* in Catalyst. For this tutorial, we will consider an example model from *Wilhelm (2009)*. The model consists of two species, *X* and *Y*, which react with each other:


In [ ]:
using Catalyst
wilhelm_model = @reaction_network begin
    k1, Y --> 2X
    k2, 2X --> X + Y
    k3, X + Y --> Y
    k4, X --> 0
    k5, 0 --> X
end

First, we simulate the model for a specific parameter set using standard workflows. We note that, after a transient, the system settles into a *steady state*.


In [ ]:
using OrdinaryDiffEq, Plots
ps = [:k1 => 12.0, :k2 => 1.0, :k3 => 1.0, :k4 => 1.5, :k5 => 1.25]
u0 = [:X => 4.0, :Y => 0.5]
oprob = ODEProblem(wilhelm_model, u0, 40.0, ps)
sol = solve(oprob)
plot(sol)

Many systems will eventually settle into steady states, and for many types of analysis it is useful to compute them directly. While it would be possible to simulate the model for a long time and use the final time point, Catalyst can do this automatically by defining and solving a `SteadyStateProblem`. Here we do so, while also importing the required [NonlinearSolve.jl](https://github.com/SciML/NonlinearSolve.jl) and [SteadyStateDiffEq.jl](https://github.com/SciML/SteadyStateDiffEq.jl) packages.


In [ ]:
using NonlinearSolve, SteadyStateDiffEq
u0 = [:X => 4.0, :Y => 0.5]
ssprob = SteadyStateProblem(wilhelm_model, u0, ps)
sol = solve(ssprob)

However, it is also possible to compute steady states directly, without first running an ODE simulation. Here, we set the ODE left-hand side to 0, creating a nonlinear system of equations:


In [ ]:
ss_ode_model(wilhelm_model)

In practice, we can solve this by creating a `NonlinearProblem` and calling `solve`. Here, the `u` we provide is not a simulation initial condition, but rather a starting point for a Newton-type algorithm.


In [ ]:
u_guess = [:X => 4.0, :Y => 0.5]
nlprob = NonlinearProblem(wilhelm_model, u_guess, ps)
sol = NonlinearSolve.solve(nlprob)

Since the initial guess is not tied to the simulation, in principle we should be able to provide any values.


In [ ]:
u_guess = [:X => 1.0, :Y => 1.0]
nlprob = NonlinearProblem(wilhelm_model, u_guess, ps)
sol = NonlinearSolve.solve(nlprob)

Here we note, however, that we find a different steady state, showing that the model has multiple steady states. This is a problem, since the two approaches described so far only find single steady states. While it would be possible to repeat them for a large number of alternative initial guesses and hope that one finds all steady states, Catalyst supports a better alternative. Since almost all chemical reaction network models correspond to (possibly rational) polynomial equations, *homotopy continuation* can be used to find steady states. Here we import the [HomotopyContinuation.jl](https://github.com/JuliaHomotopyContinuation/HomotopyContinuation.jl) package and use the `hc_steady_states` function to identify *all* steady states.


In [ ]:
import HomotopyContinuation
steady_states = hc_steady_states(wilhelm_model, ps)

We find three states in total. Typically, not all of them correspond to stable steady states. Which ones are stable can be computed with the `steady_state_stability` function.


In [ ]:
[steady_state_stability(steady_state, wilhelm_model, ps) for steady_state in steady_states]

### Part 3.2: Bifurcation
Next, we will consider bifurcation analysis. Here, we will not only compute the model's steady states, but also show how their values and properties change with the system's parameter values. Specifically, we will compute a *bifurcation diagram*, which shows a parameter's value on the x-axis and the system's steady states on the y-axis.

First, we create a `BifurcationProblem`, which bundles together some basic information about the system for which we want to compute the bifurcation diagram. Here we specify:
* The model for which we want to compute the bifurcation diagram.
* An initial guess for applying Newton's method to find a first steady state.
* The parameter set for which we wish to compute the bifurcation diagram.
* The parameter we wish to vary along the x-axis.
* The model variable we wish to plot on the y-axis. Here we will just plot the variable's concentration, but more complicated plotting formulas are also possible.


In [ ]:
using BifurcationKit
u_guess = [:X => 5.0, :Y => 2.0]
bif_par = :k1
p_start = [:k1 => 8.0, :k2 => 1.0, :k3 => 1.0, :k4 => 1.5, :k5 => 1.25]
plot_var = :X
bprob = BifurcationProblem(wilhelm_model, u_guess, p_start, bif_par; plot_var);

Next, we specify the range over which we want to vary the critical parameter. This is bundled into a `ContinuationPar` structure.


In [ ]:
p_span = (2.0, 20.0)
opts_br = ContinuationPar(p_min = p_span[1], p_max = p_span[2]);

Finally, we compute the bifurcation diagram. The resulting diagram can be plotted using the `plot` function.


In [ ]:
bif_dia = bifurcationdiagram(bprob, PALC(), 2, opts_br; bothside = true)
plot(bif_dia; xguide = bif_par, yguide = plot_var, branchlabel = "Steady state concentration")

### Part 3.3: Conservation laws 
Let us now try to apply homotopy continuation to another model.


In [ ]:
two_state_model = @reaction_network begin
    (k1,k2), X1 <--> X2
end
ps = [:k1 => 2.0, :k2 => 1.0]
hc_steady_states(two_state_model, ps)

Here we get an error because the system has conservation laws. Let's investigate this a bit more closely. Let us simulate the model for this parameter set, but with two different initial conditions.


In [ ]:
u0_1 = [:X1 => 1.0, :X2 => 1.0]
u0_2 = [:X1 => 10.0, :X2 => 10.0]
oprob_1 = ODEProblem(two_state_model, u0_1, 10.0, ps)
oprob_2 = ODEProblem(two_state_model, u0_2, 10.0, ps)
sol_1 = solve(oprob_1)
sol_2 = solve(oprob_2)
plot(plot(sol_1), plot(sol_2); lw = 4, ylims = (0.0, 15.0), size = (1000, 350))


Here we see that the steady state depends on the initial conditions. What is more, if we plot the total amount of X in the system in each case:


In [ ]:
@unpack X1, X2 = two_state_model
plot(
    plot(sol_1; idxs = [X1, X2, X1 + X2], label = ["X₁" "X₂" "X₁ + X₂ (a conserved quantity)"]), 
    plot(sol_2; idxs = [X1, X2, X1 + X2], label = ["X₁" "X₂" "X₁ + X₂ (a conserved quantity)"]); 
    lw = 4, ylims = (0.0, 15.0), size = (1000, 350)
)


We see that the total amount of *X* in the system remains constant. Next, we consider the equations used to determine steady states:


In [ ]:
ss_ode_model(two_state_model)

This is just the same equation repeated twice, i.e. we have two unknowns but only a single equation. Normally, this means that there are infinitely many solutions, and singular steady states cannot be found. However, in the simulations, we noted that the total amount of $X$ was unchanged, so we should be able to use this to find the steady states.

Here, Catalyst allows for the detection and elimination of so-called *conservation laws*. We can use the `remove_conserved = true` argument to `ss_ode_model` to automatically do this during system conversion:


In [ ]:
ss_ode_model(two_state_model; remove_conserved = true)

Here we see that we now only have a single equation and a single unknown. However, we have introduced something new, $Γ_1$, which is a parameter denoting the total amount of $X$ in the system. Since $Γ_1$ is constant (that is, *conserved*), we should be able to compute it from the initial conditions. Indeed, if we supply an initial condition to `hc_steady_states`, this is automatically used to compute $Γ_1$ and determine the system's steady states for that initial condition:


In [ ]:
hc_steady_states(two_state_model, ps; u0 = [:X1 => 1.0, :X2 => 1.0])

Conservation law elimination can be done whenever models are generated from `ReactionSystem`s. In other words, here we simulate our model as an ODE, but with the conservation law eliminated internally:


In [ ]:
u0 = [:X1 => 1.0, :X2 => 1.0]
oprob = ODEProblem(two_state_model, u0, 10.0, ps; remove_conserved = true)
sol = solve(oprob)
plot(sol)

Since $X2(0)$ can be computed from $Γ$ and $X1(0)$, this enable use to define $Γ$ isntead of $X2$ when setting our simulation conditions:

In [ ]:
ps = [:k1 => 2.0, :k2 => 1.0, :Γ => [2.0]] # `Γ` given as a vector as there might be multiple conservation law constants.
u0 = [:X1 => 1.0]
oprob = ODEProblem(two_state_model, u0, 10.0, ps; remove_conserved = true)
sol = solve(oprob)
plot(sol)